In [0]:
print("calalog and schema setup")

spark.sql("Use catalog workspace")
spark.sql("Use SCHEMA default")

print("Current catalog:",spark.sql("Select current_catalog()").first()[0])

print("Current schema:",spark.sql("Select current_schema()").first()[0])

In [0]:
#Now Read CSV
from pyspark.sql import functions as F

base_path = "/Volumes/clickstream_lab/default/raw_files"

files = [
    "batch1.csv",
    "batch2_advance_watermark.csv",
    "batch3_finalize_sessions.csv",
    "batch4_late_events.csv"
]

for file in files:
    

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{base_path}/{file}")
    )

print("Columns and data types:")
print("First 5 records:")
df.printSchema()
df.limit(5).display()   

In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/raw_files"

batch1 = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/batch1.csv")
batch2 = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/batch2.csv")
batch3 = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/batch3.csv")
batch4 = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/batch4.csv")

print("Batch 1:", batch1.count(), "rows")
print("Batch 2:", batch2.count(), "rows")
print("Batch 3:", batch3.count(), "rows")
print("Batch 4:", batch4.count(), "rows")

print("Batch 1 columns:", batch1.columns)

In [0]:
# Now we combine batch1 to batch 4
from pyspark.sql import functions as F

all_events = (
    batch1
    .unionByName(batch2)
    .unionByName(batch3)
    .unionByName(batch4)
    .withColumn(
        "event_time",
        F.to_timestamp("event_time")
    )

display(all_events.orderBy("event_time"))

In [0]:
# Now we check event timestamp
all_events.select(
    "user_id",
    "event_time",
    "event_type",
    "page_url",
    "device"
).orderBy("event_time").show(30, truncate=False)

In [0]:
#Sessionization with watermark
from pyspark.sql import functions as F

sessionized_df = (
    all_events
    .withWatermark("event_time", "10 minutes") # it manage the late event time state
    .groupBy(
        "user_id",
        F.session_window("event_time", "10 minutes") #
    )
    .agg(
        F.count("*").alias("event_count") # it count the event
    )
    .select(
        "user_id",
        F.col("session_window.start").alias("session_start"),
        F.col("session_window.end").alias("session_end"),
        "event_count"
    )
)

display(sessionized_df)

In [0]:
# Now we write session result to silver
silver_path = "/Volumes/clickstream_lab/default/raw_files/silver_sessions"

(
    sessionized_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

print("Sessionized results saved to Silver Delta storage.")

In [0]:
# Now we verfiy the silver result 
silver_df = spark.read.format("delta").load(silver_path)

display(silver_df.orderBy("user_id", "session_start"))

In [0]:
all_events.select(
    F.min("event_time").alias("earliest_event"),
    F.max("event_time").alias("latest_event"),
    F.count("*").alias("total_events")
).show(truncate=False)

In [0]:
sessionized_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("clickstream_lab.default.clickstream_silver")

In [0]:
display(
    spark.table("clickstream_lab.default.clickstream_silver")
)

In [0]:
# verify the silver table
display(
    spark.table("clickstream_lab.default.clickstream_silver")
)

In [0]:
# Now we create streming query for the late event run 
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/raw_files"

stream_df = (
    spark.readStream
    .schema(batch1.schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .csv(base_path)
    .withColumn("event_time", F.to_timestamp("event_time"))
)

sessionized_stream = (
    stream_df
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        "user_id",
        F.session_window("event_time", "10 minutes")
    )
    .agg(F.count("*").alias("event_count"))
    .select(
        "user_id",
        F.col("session_window.start").alias("session_start"),
        F.col("session_window.end").alias("session_end"),
        "event_count"
    )

In [0]:
#start the stream
query = (
    sessionized_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/raw_files/checkpoints/sessionization_test"
    )
    .trigger(availableNow=True)
    .toTable("clickstream_lab.default.clickstream_silver")
)

In [0]:
#steam-static joins
from pyspark.sql import functions as F

device_lookup = spark.createDataFrame(
    [
        ("mobile", "Mobile Device"),
        ("desktop", "Desktop Computer"),
        ("tablet", "Tablet Device")
    ],
    ["device", "device_description"]
)

enriched_events = batch1.join(
    device_lookup,
    on="device",
    how="left"
)

display(enriched_events)

In [0]:
# Save the silver in delta table
(
    sessionized_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clickstream_lab.default.clickstream_silver")
)

In [0]:
#Veify it save or not
display(
    spark.table("clickstream_lab.default.clickstream_silver")
)

In [0]:
# Check streaming query status
print("Streaming query active:", query.isActive)
print("Streaming query status:", query.status)